In [ ]:
# Part 1

# Import function from geopy
from geopy.geocoders import Nominatim

# Initialize the geocoder with a unique app name.
# Review the Geocoding section to learn more about the user_agent parameter.
geolocator = Nominatim(user_agent="sds_euro_logistics_edunsk")

# Define origin and destination addresses
origin_address = "Spitalgasse 47-51, 3001 Bern, Switzerland"
#dest_address = "Corso del Lavoro e della Scienza, 3, 38122 Trento TN, Italy"
dest_address = "Wietstruk, 19, 21727 Estorf, Germany"
#dest_address = "Schoorenstrasse, 7c, 8713 Uerikon, Switzerland"

# Call geolocator function for both addresses
origin_loc = geolocator.geocode(origin_address)
dest_loc = geolocator.geocode(dest_address)

# Get coordinates from location
origin_coords = (origin_loc.latitude, origin_loc.longitude)
dest_coords = (dest_loc.latitude, dest_loc.longitude)

# Print coordinates
print(f"Origin: {origin_coords}")
print(f"Destination: {dest_coords}")

# Print official address
print(f"Address: {dest_loc.address}")


In [ ]:
# Part 2

# Import function from geopy
from geopy import distance

# Calculate geodesic distance
baseline_km = distance.geodesic(origin_coords, dest_coords).km

# Print distance
print(f"Geodesic distance: {baseline_km:.1f} km")


In [ ]:
# Part 3

# Set ORS API Key as variable
ORS_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjA0NTY3NjI2OGNmZjQwNjZiZGM1ODFlYjU1NjMyZjE4IiwiaCI6Im11cm11cjY0In0="


In [ ]:
# Part 4

# Import request function
import requests

# Build the parameters dictionary
# Note: ORS requires coords as "longitude,latitude"
parameters = {
	"api_key": ORS_API_KEY,
	"start": f"{origin_coords[1]},{origin_coords[0]}",
	"end": f"{dest_coords[1]},{dest_coords[0]}",
}

# Define the API endpoint for car driving directions
api_url = "https://api.openrouteservice.org/v2/directions/driving-car"

# Send the request
drive_response = requests.get(api_url, params=parameters)

# Check success then calculate and print data if successful
if drive_response.status_code == 200:

	# Convert to dictionary
	drive_data = drive_response.json()

	# Extract data summary
	drive_summary = drive_data['features'][0]['properties']['summary']

	# Get driving distance and convert to kilometres
	drive_dist = drive_summary['distance']
	drive_dist_km = drive_dist / 1000

	# Get driving duration and convert to hours and minutes
	drive_dur = drive_summary['duration']
	drive_part_h = int(drive_dur // 3600)
	drive_part_min = int(drive_dur % 3600 // 60)

	# Print distance and duration
	print(f"Distance: {drive_dist_km:.1f} km")
	print(f"Duration: {drive_part_h} h {drive_part_min} min")

	# Calculate detour factor between drive and direct
	detour_factor = drive_dist_km / baseline_km

	# Print detour factor
	print(f"Detour factor: {detour_factor:.2f}")

# Set up failure case
else:
	print("Request failed.")
	print("Reason:", drive_response.text)


In [ ]:
# Part 5

# Define API endpoint for weather
meteo_url = "https://api.open-meteo.com/v1/forecast"

# Set up destination parameters
weather_params = {
	"latitude": dest_coords[0],
	"longitude": dest_coords[1],
	"current_weather": "true"
}

# Make API request
weather_response = requests.get(meteo_url, params = weather_params)

# Check success then calculate and print data if successful
if weather_response.status_code == 200:

	# Convert to dictionary
	weather_data = weather_response.json()

	# Extract temperature
	weather_temp = weather_data['current_weather']['temperature']

	# Print temperature
	print(f"Current temperature at destination: {weather_temp}°C")

# Set up failure case
else:
	print("Request failed.")
	print("Reason:", drive_response.text)


In [ ]:
# Part 6

# A list of international delivery destination coordinates (lat, lon)
deliveries = {
	"Prague": (50.0755, 14.4378),
	"Ljubljana": (46.0569, 14.5058),
	"Girona": (41.9794, 2.8214),
	"Plymouth": (50.3755, -4.1427),
	"Quimper": (47.9975, -4.0979),
	"Odense": (55.4038, 10.4024),
}

# The origin is always Bern
origin_lon = 7.4474
origin_lat = 46.9480
start_string = f"{origin_lon},{origin_lat}"

# Import functions
import time
from tqdm import tqdm
import requests

# Define API endpoints for driving directions and weather
ors_url = 'https://api.openrouteservice.org/v2/directions/driving-car'
meteo_url = "https://api.open-meteo.com/v1/forecast"

# Set up for loop
for city, coords in tqdm(deliveries.items()):
	# Set up route params
	end_string = f"{coords[1]},{coords[0]}"
	route_params = {
		'api_key': ORS_API_KEY,
		'start': start_string,
		'end': end_string
	}

	# Make API request
	route_response = requests.get(ors_url, params=route_params)

	# Check success then calculate and print data if successful
	if route_response.status_code == 200:
		
		# Convert to dictionary
		route_data = route_response.json()

		# Extract distance and duration
		route_summary = route_data['features'][0]['properties']['summary']

		# Get driving distance and convert to kilometres
		route_dist = route_summary['distance']
		route_dist_km = route_dist / 1000

		# Get driving duration and convert to hours and minutes
		route_dur = route_summary['duration']
		route_part_h = int(route_dur // 3600)
		route_part_min = int(route_dur % 3600 // 60)

		# Set up destination parameters
		weather_params = {
			"latitude": dest_coords[0],
			"longitude": dest_coords[1],
			"current_weather": "true"
		}

		# Make API request
		weather_response = requests.get(meteo_url, params = weather_params)

		# Check success then calculate and print data if successful
		if weather_response.status_code == 200:

			# Convert to dictionary
			weather_data = weather_response.json()

			# Extract temperature
			weather_temp = weather_data['current_weather']['temperature']

			# Print temperature
			print(f"\nDelivery to {city}:")
			print(f"  Drive: {route_dist_km:.1f} km ({route_part_h}h {route_part_min}min)")
			print(f"  Current Weather: {weather_temp}°C")

		# Set up failure case for weather
		else:
			print(f"\nDelivery to {city}:")
			print(f"  Drive: {route_dist_km:.1f} km ({route_part_h}h {route_part_min}min)")
			print(f"\nFailed to fetch weather for {city}.")

	# Set up failure case for drive
	else:
		print(f"\nFailed to calculate route for {city}.")

	# Rate limit the loop (2 seconds is safe for both APIs)
	time.sleep(2)
	